# POLVIFIDEL — full pipeline runner

Runs every `.py` script in order with **phase checkpoints**: each phase records itself in `data/.pipeline_status.json` when it finishes, so a re-run skips completed phases. Inside a phase, the scripts are themselves checkpointed (already-done images/rows are skipped).

**Before running:**
1. **Runtime → Change runtime type → GPU** (Phases 2–5 need it).
2. Add a Colab Secret (🔑): `AI_SANDBOX_KEY` (Princeton AI Sandbox / Portkey).
3. Check `PROJ_ROOT` in the setup cell.
4. **Runtime → Run all** — or run phases one at a time.

If a run stopped, run the **Check progress** cell to see exactly where.

In [ ]:
%%capture
!pip install -q insightface onnxruntime portkey-ai rouge-score hdbscan bertopic gensim pycocoevalcap
!python -m spacy download en_core_web_sm
import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
print('deps ready')

In [3]:
import os, subprocess, sys
from pathlib import Path
from datetime import datetime
import json as _json
from google.colab import drive, userdata

drive.mount('/content/drive')

# UPDATE if your Drive path differs
PROJ_ROOT = Path('/content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL')

SCRIPTS     = PROJ_ROOT / 'scripts'
REFS        = str(PROJ_ROOT / 'data' / 'df_corpus.csv')
RECOLLECTED = str(PROJ_ROOT / 'data' / 'recollected')
METRICS     = str(PROJ_ROOT / 'data' / 'metrics' / 'metrics.csv')
STATUS_PATH = PROJ_ROOT / 'data' / '.pipeline_status.json'

os.environ['POLVIFIDEL_ROOT'] = str(PROJ_ROOT)
# Princeton AI Sandbox via Portkey -- ONE key for all API models (GPT + Gemini).
try:
    os.environ['AI_SANDBOX_KEY'] = userdata.get('AI_SANDBOX_KEY')
except Exception:
    raise RuntimeError('Add AI_SANDBOX_KEY to Colab Secrets (key icon) and grant this notebook access.')

CAPTION_MODELS = 'all'   # 'all' (gpt4o + gemini) or 'gpt4o'
FORCE = set()            # e.g. {'Phase 3'} to force-rerun an already-completed phase

def run(script, *args):
    cmd = [sys.executable, '-u', script, *map(str, args)]   # -u = unbuffered, so progress prints live
    print(); print('>>>', *cmd, flush=True)
    r = subprocess.run(cmd, cwd=str(SCRIPTS))
    if r.returncode != 0:
        raise RuntimeError(script + ' failed (exit ' + str(r.returncode) + ')')
    print('<<<', script, 'done', flush=True)

def _status():
    try:
        return _json.loads(STATUS_PATH.read_text())
    except Exception:
        return {}

def phase(name, *steps):
    '''Run (script, *args) steps unless this phase is already marked complete.'''
    st = _status(); done = st.get('done', {})
    if name in done and not any(f.lower() in name.lower() for f in FORCE):
        print('==', name, '-- SKIP (completed', done[name], ') ==')
        return
    print('==', name, '-- running ==')
    for step in steps:
        run(*step)
    done[name] = datetime.now().isoformat(timespec='seconds')
    st['done'] = done
    STATUS_PATH.write_text(_json.dumps(st, indent=2))
    print('==', name, '-- COMPLETE ==')

def progress():
    D = PROJ_ROOT / 'data'
    def rows(p):
        try:
            return sum(1 for _ in open(p)) - 1
        except Exception:
            return 0
    n_img = len(list((D / 'images').glob('*.jpg')))
    n_det = len(list((D / 'detections').glob('*.json')))
    cap_rows = sum(rows(p) for p in (D / 'captions').glob('*.csv'))
    done = _status().get('done', {})
    checks = [
        ('Phase 1 - Corpus',     (D / 'df_corpus.csv').exists(), str(rows(D / 'df_corpus.csv')) + ' imgs in manifest'),
        ('Phase 2 - Gallery',    (D / 'gallery' / 'embeddings.pkl').exists(), 'embeddings.pkl built'),
        ('Phase 3 - Detection',  n_det > 0, str(n_det) + '/' + str(n_img) + ' images detected'),
        ('Phase 4 - Dictionary', (D / 'entity_dict.json').exists(), 'entity_dict.json built'),
        ('Phase 5 - Captions',   cap_rows > 0, str(cap_rows) + ' caption rows'),
        ('Phase 6 - Metrics',    (D / 'metrics' / 'metrics.csv').exists(), str(rows(D / 'metrics' / 'metrics.csv')) + ' metric rows'),
        ('Phase 7 - Analyses',   (D / 'metrics' / 'ner_eval.csv').exists() and (D / 'metrics' / 'topic_results.csv').exists(), 'ner_eval + topic_results'),
    ]
    print('Pipeline progress (artifacts on Drive):')
    for name, ok, info in checks:
        flag = 'DONE' if name in done else ('hasOut' if ok else 'pending')
        print('  [' + flag + ']', name, '-', info if ok else 'not yet')
    print()
    print('  DONE = marked complete | hasOut = artifacts present, not marked | pending = nothing yet')
    print('  -> a stopped run halted at the first non-DONE phase; re-run all and completed phases skip.')

import torch
print('Project:', PROJ_ROOT, '| GPU:', torch.cuda.is_available())
progress()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL | GPU: True
Pipeline progress (artifacts on Drive):
  [DONE] Phase 1 - Corpus - 830 imgs in manifest
  [DONE] Phase 2 - Gallery - embeddings.pkl built
  [DONE] Phase 3 - Detection - 1351/1351 images detected
  [DONE] Phase 4 - Dictionary - entity_dict.json built
  [DONE] Phase 5 - Captions - 6046 caption rows
  [hasOut] Phase 6 - Metrics - 6028 metric rows
  [pending] Phase 7 - Analyses - not yet

  DONE = marked complete | hasOut = artifacts present, not marked | pending = nothing yet
  -> a stopped run halted at the first non-DONE phase; re-run all and completed phases skip.


## Phase 0 — reset stale pilot artifacts
Pilot outputs (20-image captions/detections + a saturated, old-schema `metrics.csv`) would break or contaminate the full run. Set `RESET_STALE = True` once for a clean slate. (Leaves images, gallery, df_corpus, recollected, and the status file intact.)

In [ ]:
RESET_STALE = False   # set True ONCE to clear pilot artifacts; keep False afterwards (it deletes detections/captions!)

if RESET_STALE:
    import json
    mt = PROJ_ROOT / 'data' / 'metrics' / 'metrics.csv'
    if mt.exists():
        mt.unlink(); print('removed', mt.name)
    for sub in ['detections', 'captions']:
        d = PROJ_ROOT / 'data' / sub
        n = sum(1 for f in d.glob('*') if f.is_file() and (f.unlink() or True))
        print('cleared', n, 'files from data/' + sub + '/')
    # IMPORTANT: clearing detections/captions invalidates these phases, so un-mark
    # them in the checkpoint -- otherwise they are SKIPPED and never regenerate
    # (that bug left detections empty -> vifidel/polvifidel all NaN).
    s = _status()
    for k in list(s.get('done', {})):
        if any(x in k for x in ['Detection', 'Dictionary', 'Captions', 'Metrics', 'Analyses']):
            s['done'].pop(k, None)
    STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)   # avoid FileNotFoundError on Drive
    STATUS_PATH.write_text(json.dumps(s, indent=2))
    print('Un-marked dependent phases (Detection..Analyses) so they regenerate.')
    print('Stale artifacts cleared.')
else:
    print('RESET_STALE = False - keeping existing artifacts.')

cleared 40 files from data/detections/
cleared 0 files from data/captions/
Un-marked dependent phases (Detection..Analyses) so they regenerate.
Stale artifacts cleared.


## Check progress / where did it stop?
Run this anytime — it inspects the artifacts on Drive and the status file to show which phases are complete and where a stopped run left off.

In [ ]:
progress()

## Phase 1 — Corpus
Re-mine archive → LLM-classify mass cells → consolidate → download images.

In [ ]:
phase('Phase 1 - Corpus',
      ('01a_recollect_from_archive.py',),
      ('01c_classify_mass_cells.py',),
      ('01b_consolidate_corpus.py', '--input-dir', RECOLLECTED),
      ('download_corpus.py',))

## Phase 2 — Face gallery (GPU)

In [ ]:
phase('Phase 2 - Gallery',
      ('build_face_gallery.py', '--report'),
      ('build_face_gallery.py',))

## Phase 3 — Entity detection (GPU)

In [ ]:
phase('Phase 3 - Detection',
      ('02_detect_entities.py',))

== Phase 3 - Detection -- running ==

>>> /usr/bin/python3 -u 02_detect_entities.py
<<< 02_detect_entities.py done
== Phase 3 - Detection -- COMPLETE ==


## Phase 4 — Entity dictionary
Needs `OPENAI_API_KEY` (or add `--no-llm`).

In [ ]:
phase('Phase 4 - Dictionary',
      ('03_build_entity_dictionary.py', '--from-detections'))

== Phase 4 - Dictionary -- running ==

>>> /usr/bin/python3 -u 03_build_entity_dictionary.py --from-detections
<<< 03_build_entity_dictionary.py done
== Phase 4 - Dictionary -- COMPLETE ==


## (Optional) Fast floor-effect check — does prompting move the metrics?

Runs the whole pipeline **end-to-end on ~40 corpus images** (detect → dictionary → caption → metrics → summary) in ~5 minutes, so you can see whether the automated metrics improve across the four prompting conditions **before** committing to the full run. It detects just these 40 (no need to wait for full Phase 3), uses `run()` (not `phase()`) so it marks nothing done, and the full phases below reuse this work. In the `pilot_summary` table, check that `polvifidel`/`vifidel` have real numbers and that the `Δ vs baseline` rows are non-zero. If they're flat, stop and tune before the full run.

In [ ]:
# FAST floor-effect check: detect -> dict -> caption -> score ~40 corpus images end-to-end (~5 min).
# Self-contained: detects just these 40 (no need to wait for full Phase 3). Uses run() (NOT phase()),
# so it does NOT mark any phase done; the full phases below still run and reuse this work.
SMOKE_N = 40
run('02_detect_entities.py', '--references', REFS, '--limit', str(SMOKE_N))
run('03_build_entity_dictionary.py', '--from-detections')
run('04_generate_captions_api.py', '--model', CAPTION_MODELS, '--references', REFS, '--limit', str(SMOKE_N))
run('05_compute_metrics.py', '--references', REFS, '--skip-spice')
run('pilot_summary.py', '--metrics', METRICS)


>>> /usr/bin/python3 -u 02_detect_entities.py --references /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv --limit 40
<<< 02_detect_entities.py done

>>> /usr/bin/python3 -u 03_build_entity_dictionary.py --from-detections
<<< 03_build_entity_dictionary.py done

>>> /usr/bin/python3 -u 04_generate_captions_api.py --model all --references /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv --limit 40
<<< 04_generate_captions_api.py done

>>> /usr/bin/python3 -u 05_compute_metrics.py --references /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv --skip-spice
<<< 05_compute_metrics.py done

>>> /usr/bin/python3 -u pilot_summary.py --metrics /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/metrics/metrics.csv
<<< pilot_summary.py done


## Phase 5 — Caption generation (GPU + API keys)

In [ ]:
phase('Phase 5 - Captions',
      ('04_generate_captions_api.py', '--model', CAPTION_MODELS, '--references', REFS))

== Phase 5 - Captions -- running ==

>>> /usr/bin/python3 -u 04_generate_captions_api.py --model all --references /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv


## Phase 5b — Open-source VLMs (InternVL2-8B, Qwen2.5-VL)  *(A100 only)*
Large models — do **not** run on a T4. On an A100 runtime set `RUN_OPENSOURCE = True`.
Uses `run()` (not `phase()`), so it won't touch the checkpoint; it appends to the same
`data/captions/` files as the API route, so Phase 6 metrics pick them up automatically.

In [ ]:
RUN_OPENSOURCE = False   # set True on an A100 runtime to caption with InternVL + Qwen
if RUN_OPENSOURCE:
    # extra deps for the open-source VLMs (only needed when actually running them)
    !pip install -q "transformers>=4.49" accelerate einops timm torchvision
    run('04_generate_captions_cluster.py', '--model', 'all', '--references', REFS)
else:
    print('RUN_OPENSOURCE=False -- skipping InternVL/Qwen (set True on an A100).')

## Phase 6 — Metrics + floor-effect check
Confirm POLVIFIDEL varies across conditions in the summary.

In [ ]:
phase('Phase 6 - Metrics',
      ('05_compute_metrics.py', '--references', REFS, '--skip-spice'),
      ('pilot_summary.py', '--metrics', METRICS))

== Phase 6 - Metrics -- running ==

>>> /usr/bin/python3 -u 05_compute_metrics.py --references /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv --skip-spice
<<< 05_compute_metrics.py done

>>> /usr/bin/python3 -u pilot_summary.py --metrics /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/metrics/metrics.csv
<<< pilot_summary.py done
== Phase 6 - Metrics -- COMPLETE ==


## Phase 7 — Analyses (the four poster results)

In [4]:
phase('Phase 7 - Analyses',
      ('06b_metric_correlation.py',),
      ('08_ner_evaluation.py', '--references', REFS),
      ('07_topic_modeling.py', '--labels', REFS, '--refs', REFS))
print('Pipeline complete.')

== Phase 7 - Analyses -- running ==

>>> /usr/bin/python3 -u 06b_metric_correlation.py
<<< 06b_metric_correlation.py done

>>> /usr/bin/python3 -u 08_ner_evaluation.py --references /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv
<<< 08_ner_evaluation.py done

>>> /usr/bin/python3 -u 07_topic_modeling.py --labels /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv --refs /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/df_corpus.csv
<<< 07_topic_modeling.py done
== Phase 7 - Analyses -- COMPLETE ==
Pipeline complete.


In [6]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, '07_topic_modeling.py', '--labels', REFS, '--refs', REFS],
    cwd=str(SCRIPTS), capture_output=True, text=True)
print("RETURN CODE:", r.returncode)
print("----- STDOUT (tail) -----"); print(r.stdout[-2000:])
print("----- STDERR (tail) -----"); print(r.stderr[-4000:])


RETURN CODE: 1
----- STDOUT (tail) -----
  nyt_reference: 13 topics, 182/830 noise docs
  gemini_annotated_aux: 4 topics, 121/728 noise docs
  gemini_baseline: 16 topics, 61/830 noise docs
  gemini_textual_aux: 13 topics, 76/728 noise docs
  gemini_visual_aux: 17 topics, 134/728 noise docs

----- STDERR (tail) -----
2026-06-24 14:52:34.392262: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Topic modeling runs:   0%|          | 0/9 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7847.94it/s]

Topic modeling runs:  11%|█         | 1/9 [00:24<03:15, 24.48s/it]

Loading weights: 100%|███